# CHEM 269 — Tier-2 CREST+ALPB Pipeline

**Purpose:** Run dual-dielectric CREST conformer sampling on a stratified batch of CycPeptMPDB compounds.

**Before running:**
1. Upload `colab_utils.py` and your assigned `batch_N.csv` to Google Drive
2. Set `BATCH_CSV` and `BATCH_NAME` in the Config cell to match your batch
3. Run all cells top to bottom — the install cell will restart the runtime automatically
4. After restart, run from Cell 3 onwards (Cell 1 & 2 only need to run once per session)

**Each Colab account should run 2 notebooks simultaneously, each with a different batch.**

In [ ]:
# ── CELL 1: Install condacolab (runtime will restart automatically) ──────────
# Only run this once per session. After restart, skip to Cell 3.
import sys

try:
    import condacolab
    print('condacolab already installed')
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()  # <-- triggers runtime restart

In [ ]:
# ── CELL 2: Install CREST, xtb, RDKit (run after restart) ───────────────────
# This takes ~5-8 minutes. Only needed once per session.
import subprocess, sys

print('Installing crest + xtb + rdkit via mamba...')
subprocess.run(
    ['mamba', 'install', '-c', 'conda-forge', 'crest', 'xtb', 'rdkit', 'tqdm', '-y', '-q'],
    check=True
)

# Verify
r = subprocess.run(['crest', '--version'], capture_output=True, text=True)
print('CREST:', r.stdout.strip() or r.stderr.strip())
r = subprocess.run(['xtb', '--version'], capture_output=True, text=True)
print('xtb:  ', r.stdout.strip()[:60])
import rdkit; print('RDKit:', rdkit.__version__)

In [ ]:
# ── CELL 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# ── CELL 4: Configuration — EDIT THESE BEFORE RUNNING ────────────────────────

# Path to your batch CSV on Google Drive
# Change batch_1 to batch_2, batch_3, or batch_4 for each notebook
BATCH_CSV   = '/content/drive/MyDrive/chem269_tier2/batch_1.csv'
BATCH_NAME  = 'batch_1'   # used for naming the results file

# Path to colab_utils.py on Google Drive
UTILS_PATH  = '/content/drive/MyDrive/chem269_tier2/colab_utils.py'

# Results will be saved here — one CSV per batch, appended after each compound
RESULTS_DIR = '/content/drive/MyDrive/chem269_tier2/results/'

# CREST threads — use 4 for Colab Pro (up to 8 on Pro+)
N_THREADS   = 4

# Temp working directory for CREST runs (local, not Drive — faster I/O)
WORK_ROOT   = '/tmp/crest_runs'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(WORK_ROOT, exist_ok=True)
print(f'Config set: {BATCH_NAME}, results -> {RESULTS_DIR}')

In [ ]:
# ── CELL 5: Import colab_utils.py from Drive ──────────────────────────────────
import importlib.util, sys

spec = importlib.util.spec_from_file_location('colab_utils', UTILS_PATH)
colab_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_utils)
sys.modules['colab_utils'] = colab_utils

print('colab_utils loaded successfully')
print('Functions available:', [f for f in dir(colab_utils) if not f.startswith('_')])

In [ ]:
# ── CELL 6: Load batch and check for already-completed compounds ──────────────
import pandas as pd
from pathlib import Path

df = pd.read_csv(BATCH_CSV)
print(f'Batch {BATCH_NAME}: {len(df)} compounds')
print(f'PAMPA range: {df["PAMPA"].min():.2f} to {df["PAMPA"].max():.2f}')
print(df[['ID','PAMPA']].head(10).to_string(index=False))

RESULTS_PATH = Path(RESULTS_DIR) / f'tier2_crest_{BATCH_NAME}.csv'

done_ids = set()
if RESULTS_PATH.exists():
    done_df  = pd.read_csv(RESULTS_PATH)
    done_ids = set(done_df['ID'].astype(str))
    print(f'\nResuming: {len(done_ids)} compounds already completed')
else:
    print('\nStarting fresh — no existing results found')

remaining = df[~df['ID'].astype(str).isin(done_ids)]
print(f'To process: {len(remaining)} compounds')

In [ ]:
# ── CELL 7: Main processing loop ─────────────────────────────────────────────
# Runs CREST+ALPB for each compound. Saves to Drive after every compound.
# Safe to interrupt and resume — already-completed compounds are skipped.

import time
from pathlib import Path

RESULTS_PATH = Path(RESULTS_DIR) / f'tier2_crest_{BATCH_NAME}.csv'
n_total      = len(remaining)
n_done       = 0
n_failed     = 0
t_start      = time.time()

print(f'Starting Tier-2 CREST on {n_total} compounds with {N_THREADS} threads each')
print(f'Results saved to: {RESULTS_PATH}')
print('-' * 60)

for _, row in remaining.iterrows():
    mol_id = row['ID']
    smiles = row['SMILES_canonical']
    t_mol  = time.time()

    result = colab_utils.process_compound_crest(
        mol_id    = mol_id,
        smiles    = smiles,
        n_threads = N_THREADS,
        work_root = WORK_ROOT,
    )

    # Add PAMPA for convenience
    result['PAMPA'] = row['PAMPA']

    # Append to Drive CSV immediately
    result_row = pd.DataFrame([result])
    result_row.to_csv(
        RESULTS_PATH,
        mode='a',
        header=not RESULTS_PATH.exists(),
        index=False,
    )

    n_done += 1
    elapsed  = time.time() - t_mol
    total_el = time.time() - t_start
    avg_time = total_el / n_done
    eta_h    = avg_time * (n_total - n_done) / 3600

    status = result.get('error') or 'OK'
    if status != 'OK':
        n_failed += 1

    dpsa = result.get('delta_psa3d', float('nan'))
    mem  = result.get('mem_psa3d',   float('nan'))
    print(
        f'[{n_done:4d}/{n_total}] ID={mol_id:<6} '
        f'ΔPSA={dpsa:>7.1f}  mem_psa={mem:>7.1f}  '
        f't={elapsed:>5.0f}s  ETA={eta_h:.1f}h  status={status}'
    )

print(f'\nDone. {n_done} processed, {n_failed} failed.')
print(f'Results: {RESULTS_PATH}')

In [ ]:
# ── CELL 8: Results summary ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_PATH = Path(RESULTS_DIR) / f'tier2_crest_{BATCH_NAME}.csv'

if not RESULTS_PATH.exists():
    print('No results file yet — run the main loop first')
else:
    res = pd.read_csv(RESULTS_PATH)
    ok  = res[res['error'].isna()]

    print(f'Total processed: {len(res)}')
    print(f'Successful:      {len(ok)}')
    print(f'Failed:          {res["error"].notna().sum()}')
    if res['error'].notna().any():
        print('\nFailure breakdown:')
        print(res['error'].value_counts().to_string())

    if len(ok) > 0:
        print('\n--- Key descriptor statistics ---')
        for col in ['mem_psa3d', 'aq_psa3d', 'delta_psa3d', 'delta_hb']:
            if col in ok.columns:
                v = ok[col].dropna()
                print(f'{col:<18}: mean={v.mean():.1f}  std={v.std():.1f}  '
                      f'min={v.min():.1f}  max={v.max():.1f}')

        print('\n--- Permeable (PAMPA > -6.0) vs Impermeable ---')
        ok['permeable'] = ok['PAMPA'] > -6.0
        for col in ['mem_psa3d', 'delta_psa3d']:
            if col in ok.columns:
                grp = ok.groupby('permeable')[col].mean()
                print(f'{col}: permeable={grp.get(True, float("nan")):.1f}  '
                      f'impermeable={grp.get(False, float("nan")):.1f}')

In [ ]:
# ── CELL 9: Download results to local machine (optional) ──────────────────────
from google.colab import files
from pathlib import Path

RESULTS_PATH = Path(RESULTS_DIR) / f'tier2_crest_{BATCH_NAME}.csv'

if RESULTS_PATH.exists():
    files.download(str(RESULTS_PATH))
    print(f'Downloaded: {RESULTS_PATH.name}')
else:
    print('No results file to download yet')